## Part 1 - Predicting Iris Species with PyTorch

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load dataset
iris = load_iris()
X = iris.data
y = iris.target

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

# Define model
class IrisNet(nn.Module):
    def __init__(self):
        super(IrisNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )

    def forward(self, x):
        return self.model(x)

model = IrisNet()

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    test_outputs = model(X_test)
    _, predicted = torch.max(test_outputs, 1)
    accuracy = (predicted == y_test).sum().item() / y_test.size(0)

print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

Epoch [10/100], Loss: 0.7564
Epoch [20/100], Loss: 0.3915
Epoch [30/100], Loss: 0.2535
Epoch [40/100], Loss: 0.1659
Epoch [50/100], Loss: 0.0843
Epoch [60/100], Loss: 0.0473
Epoch [70/100], Loss: 0.0375
Epoch [80/100], Loss: 0.0324
Epoch [90/100], Loss: 0.0301
Epoch [100/100], Loss: 0.0276

Test Accuracy: 96.67%


## Part 2 - Estimating Causal Effects with Do-Calculus

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.linear_model import LinearRegression

# Load data
iris = load_iris(as_frame=True)
df = iris.frame

df["species"] = df["target"]
df = df.rename(columns={
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width"
})

In [3]:
# One-hot encode species (confounder)
df = pd.get_dummies(df, columns=["species"], drop_first=False)

# Fit outcome model Y ~ X + S
features = ["petal_length"] + [c for c in df.columns if "species_" in c]
model = LinearRegression()
model.fit(df[features], df["petal_width"])

LinearRegression()

In [4]:
# Estimate E[Y | do(X=x)]
def do_intervention(x_value):
    expected = 0
    species_cols = [c for c in df.columns if "species_" in c]

    # Empirical P(S=s)
    species_probs = df[species_cols].mean()

    for s in species_cols:
        # Create input vector
        row = {col: 0 for col in features}
        row["petal_length"] = x_value
        row[s] = 1

        X_input = np.array([row[col] for col in features]).reshape(1, -1)
        y_pred = model.predict(X_input)[0]

        expected += y_pred * species_probs[s]

    return expected

In [5]:
# Example intervention
x_val = 4.0
print(f"E[Petal Width | do(Petal Length = {x_val})] = {do_intervention(x_val):.3f}")

E[Petal Width | do(Petal Length = 4.0)] = 1.255


/Library/Python/3.9/site-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/Library/Python/3.9/site-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/Library/Python/3.9/site-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
